# Dataset et DataLoader : regrouper des exemples de même forme

Retrouver les mêmes paires mesure/cible, puis les parcourir par petits lots.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#dataset-et-batches).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La première boucle travaillait sur tout le petit tableau. Pour un corpus plus grand, nous voulons retrouver un exemple et n’en charger que quelques-uns à la fois.

Nous gardons exactement les trois extraits fictifs du socle. Leurs valeurs sont choisies à la main, pas mesurées sur les fichiers audio. Chaque extrait possède deux descripteurs.

| Indice | RMS choisi | Facteur de crête choisi | Cible |
| --- | --- | --- | --- |
| 0 | 0,10 | 1,5 | 0 : voix |
| 1 | 0,20 | 4,5 | 1 : percussion |
| 2 | 0,15 | 2,5 | 0 : voix |

mesures est de forme (3,2) : extraits, puis descripteurs. cibles est de forme (3,) : un indice de classe par extrait.

Un Dataset décrit comment obtenir une paire. Notre classe Extraits conserve les deux tableaux ; elle ne calcule aucun nouveau descripteur et ne mélange pas encore les données.

## Démonstration 1 · Du tableau au premier lot

Lire séparément le contrat d’un exemple et le parcours des exemples.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader


### Calculer l’état suivant

mesures et cibles sont les tableaux déjà connus. Le type long convient ici aux indices de classes utilisés par notre perte.

In [4]:
mesures = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)



In [5]:
observer('mesures', mesures)
observer('cibles', cibles)

mesures forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Définir la règle avant de l’utiliser

__init__ recevra les tableaux lors de la construction. self désigne l’instance ; self.mesures et self.cibles seront ses attributs.

Le nom à droite est l’argument reçu. L’attribut à gauche garde un accès au même tenseur ; cette affectation ne copie pas ses valeurs.

len(dataset) appelle __len__. Nous comptons les trois annotations, qui correspondent aux trois lignes de mesures.

dataset[1] appelle __getitem__(1). Les deux sélections utilisent le même indice : mesures de percussion et cible 1 restent associées.

In [6]:
class Extraits(Dataset):
    def __init__(self, mesures, cibles):
        self.mesures = mesures
        self.cibles = cibles
    def __len__(self):
        return len(self.cibles)
    def __getitem__(self, indice):
        return self.mesures[indice], self.cibles[indice]



### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [7]:
dataset = Extraits(mesures, cibles)
mesures_1, cible_1 = dataset[1]


In [8]:
observer('mesures_1', mesures_1)
observer('cible_1', cible_1)

mesures_1 forme (2,) type torch.float32 requires_grad False
  valeurs : [0.20000000298023224, 4.5]
cible_1 forme () type torch.int64 requires_grad False
  valeurs : 1


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [9]:
print("extrait 1 :", mesures_1, "cible :", cible_1.item())


extrait 1 : tensor([0.2000, 4.5000]) cible : 1


### Calculer l’état suivant

batch_size=2 demande jusqu’à deux exemples par lot. shuffle=False garde l’ordre 0,1,2 ; num_workers=0 charge dans le processus courant.

In [10]:
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)


### Calculer l’état suivant

iter(loader) crée un parcours. next(parcours) demande son premier lot. Comme toutes les lignes ont deux valeurs, l’assemblage par défaut peut les empiler.

In [11]:
parcours = iter(loader)
premier_lot = next(parcours)
entrees, annotations = premier_lot


In [12]:
observer('parcours', parcours)
observer('premier_lot', premier_lot)
observer('entrees', entrees)
observer('annotations', annotations)

premier_lot : 2 éléments ; formes des tenseurs : [(2, 2), (2,)]
entrees forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5]]
annotations forme (2,) type torch.int64 requires_grad False
  valeurs : [0, 1]


### Lire les sorties

Il ne reste que l’extrait 2 : sa forme est (1,2). batch_size est une taille maximale, pas la promesse de remplir chaque lot.

In [13]:
print("premier lot :", entrees)
print("cibles :", annotations.tolist())
print("formes :", tuple(entrees.shape), tuple(annotations.shape))


premier lot : tensor([[0.1000, 1.5000],
        [0.2000, 4.5000]])
cibles : [0, 1]
formes : (2, 2) (2,)


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
dernieres_entrees, dernieres_cibles = next(parcours)


In [15]:
observer('dernieres_entrees', dernieres_entrees)
observer('dernieres_cibles', dernieres_cibles)

dernieres_entrees forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.15000000596046448, 2.5]]
dernieres_cibles forme (1,) type torch.int64 requires_grad False
  valeurs : [0]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [16]:
print("dernier lot :", tuple(dernieres_entrees.shape), dernieres_cibles.tolist())


dernier lot : (1, 2) [0]


### Calculer l’état suivant

next(iter(loader)) réunit les deux appels. Un nouvel iter recommence un parcours ; ce n’est pas le prochain lot du parcours précédent.

Nous exécutons d’abord iter(loader), puis next sur l’objet obtenu. Pour continuer le même parcours, nous conservons cet objet dans une variable.

In [17]:
premier_de_nouveau = next(iter(loader))


In [18]:
observer('premier_de_nouveau', premier_de_nouveau)

premier_de_nouveau : 2 éléments ; formes des tenseurs : [(2, 2), (2,)]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [19]:
print("nouveau parcours :", premier_de_nouveau[1].tolist())

nouveau parcours : [0, 1]


Le chargeur modifie le regroupement et l’ordre de lecture, pas l’association entre une ligne et sa cible.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Ce jeu contient trois paires. Combien d’exemples contient-il ?

In [20]:
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_y = torch.tensor([0,1,0])
exo_dataset = torch.utils.data.TensorDataset(exo_x,exo_y)
exo_nombre = ...  # Remplacer les points par votre réponse.
if exo_nombre is Ellipsis or exo_nombre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_nombre == len(exo_dataset)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Chaque paire mesures/cible compte pour un exemple. La largeur des mesures ne compte pas les exemples.

</details>

### Exercice 2

Avec batch_size=2 et drop_last=False, quelle forme aura le dernier lot de mesures ?

In [21]:
exo_x = torch.zeros(3,2)
exo_y = torch.tensor([0,1,0])
exo_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(exo_x,exo_y),batch_size=2,shuffle=False)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(list(exo_loader)[-1][0].shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Après le premier lot de deux exemples, il reste un exemple avec ses deux caractéristiques.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_y = torch.tensor([0,1,0])
exo_dataset = torch.utils.data.TensorDataset(exo_x,exo_y)
exo_nombre = 3
if exo_nombre is Ellipsis or exo_nombre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_nombre == len(exo_dataset)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.zeros(3,2)
exo_y = torch.tensor([0,1,0])
exo_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(exo_x,exo_y),batch_size=2,shuffle=False)
exo_forme = (1,2)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(list(exo_loader)[-1][0].shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [22]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()